In [ ]:
# Источник истины: этот файл. Сборка: make sync-notebooks.

# ЛР1. Варианты 3 и 4. Ridge и выбор модели: общий эксперимент

Сегодня превращаем одну идею из лекции в собственный работающий компонент.
Данные, предобработка, визуализация и обёртки уже готовы. Сначала маленький пример, затем код и эксперимент.

## Одна ЛР1, два занятия, от 0 до 10 баллов

Это **одна лабораторная работа на два занятия и самостоятельное завершение дома**.
Вы выполняете только назначенный вариант. Начать часть A можно после лекции 2;
часть B завершайте после лекций 3–4. Два этапа не оцениваются как две разные работы.

| За что | Баллы |
|---|---:|
| Часть A: корректный базовый компонент и его проверки | 3.5 |
| Сквозной эксперимент с собственным базовым компонентом | 1.0 |
| Объяснение базового решения на защите | 0.5 |
| Часть B: расширенный контракт и его проверки | 2.5 |
| Контролируемый эксперимент и содержательный вывод | 1.5 |
| Объяснение расширения и небольшое изменение на защите | 1.0 |
| **Максимум** | **10.0** |

Законченный базовый маршрут даёт **до 5/10**, полный — **до 10/10**. Частичные баллы
внутри строки зависят от реально работающих частей; провал одного теста не обнуляет всё.

**Что сдаём:** сохранённый `.ipynb`, собственные выводы и сгенерированный файл прогнозов
на закрытой тестовой части.
Не меняйте тесты, служебные функции и фиксированное разбиение ради прохождения проверки.

## Как пользоваться заготовкой
`MODE = "base"` — часть A; `MODE = "full"` — A и B. Не ставьте `full`, пока расширение не готово.
`NotImplementedError` означает явно незавершённую работу: проверки и зависимый эксперимент
покажут «НЕ ВЫПОЛНЕНО», а не подставят библиотечное решение. Настоящие ошибки и неверные
`assert` останавливают выполнение и требуют исправления. После правок перезапустите kernel.
В разделе `Референсный baseline` готовая модель отмечена отдельно и не считается вашим кодом.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython import get_ipython
from IPython.display import display

if get_ipython() is not None:
    get_ipython().run_line_magic("matplotlib", "inline")
from sklearn.base import BaseEstimator, RegressorMixin
from sklearn.linear_model import Ridge
from sklearn.model_selection import KFold, cross_val_predict, cross_validate
from sklearn.pipeline import make_pipeline
from sklearn.utils.validation import check_is_fitted

# Готовый поиск корня: notebook можно открыть из своей подпапки.
ROOT = next(
    (
        p
        for p in [Path.cwd(), *Path.cwd().parents]
        if (
            (p / "materials/01-regression/code/course_support.py").is_file()
            and (
                p
                / "materials/02-data/datasets/support-tickets/student/tickets_lab1_dev.csv"
            ).is_file()
        )
        or (
            (p / "course_support.py").is_file()
            and (p / "data/tickets_lab1_dev.csv").is_file()
        )
    ),
    None,
)
if ROOT is None:
    raise FileNotFoundError(
        "Распакуйте весь комплект; нужны данные ЛР1 и course_support.py."
    )
SUPPORT = (
    ROOT
    if (ROOT / "course_support.py").is_file()
    else ROOT / "materials/01-regression/code"
)
if str(SUPPORT) not in sys.path:
    sys.path.insert(0, str(SUPPORT))
from course_support import (
    BLUE,
    FEATURES,
    TARGET,
    augment,
    baseline_pipeline,
    check_arrays,
    export_predictions,
    load_tables,
    make_preprocessor,
    metrics,
    mse,
    plot_residuals,
    plot_target,
    run_checks,
)

In [ ]:
VARIANT = 3
MODE = "base"
EXPORT_FINAL = False

In [ ]:
assert MODE in {"base", "full"}
checks = {}

## От знакомой задачи к более интересным данным

`маркер=DATA`

Как в первом семинаре, предсказываем `resolution_minutes` **при регистрации обращения**.
Исходный `intro_tickets.csv` сохранён без изменений: 480 строк, три числовых признака и target.
Здесь используем **новый синтетический набор**, а не дополнительные реальные обращения
и не переименованную копию исходных ответов. Словарь полей находится в паспорте набора данных.

Добавлены канал и тема обращения, число вложений, число работающих операторов,
выходной день и размер первого запроса в байтах. Есть пропуски, близкие по информации
признаки, нелинейный эффект высокой очереди и неодинаковый шум.

Фиксированы 1440 строк train, 480 validation и 480 test.
Разбиение случайное для специально сгенерированных независимых обращений; переносить
его автоматически на реальные временные или клиентские данные нельзя.

Предобработка готова: числовые пропуски → медиана train; числовые масштабы → статистики
train; категории → готовое кодирование. Если эти компоненты пока незнакомы, используйте
их как готовую цепочку; на семинаре 2 вернёмся к их смыслу. `ticket_id` не входит в признаки.

In [ ]:
train, valid, test = load_tables(ROOT)
X_train, y_train = train[FEATURES], train[TARGET].to_numpy()
X_valid, y_valid = valid[FEATURES], valid[TARGET].to_numpy()
print("Train / validation / test:", len(train), len(valid), len(test))
display(train.head(4))
display(train[FEATURES].isna().sum().rename("Пропуски на train").to_frame())
plot_target(train)

## Модель и обозначения

$X$ — числовые входы после предобработки, $X\in\mathbb R^{n\times d}$;
$y\in\mathbb R^n$ — ответы. Внутри оптимизатора используем
$Z=[\mathbf 1\;X]$ и $\theta=(b,w_1,\ldots,w_d)^\top$.
Тогда $\hat y=Z\theta$; нулевая координата — **свободный член**, а не дополнительный измеренный признак.

$$J(\theta)=\frac1n\|Z\theta-y\|_2^2+\lambda\sum_{j=1}^{d}\theta_j^2.$$

В части A $\lambda=0$. Для частей с регуляризацией параметр `l2` означает именно $\lambda$.
MSE и $J$ измеряются здесь в квадратных минутах; MAE и RMSE — в минутах.
**Метрика проверки и критерий обучения — не одно и то же.**

## Референсный baseline — готовый, не ваша реализация
Библиотечное решение нужно для масштаба ошибки и сравнения, а не для заполнения задания.
Здесь `l2=0`, то есть нерегуляризованная линейная задача. Медианный baseline полезен по MAE,
но для MSE естественная константа — среднее train; ниже показаны обе метрики одного среднего прогноза.

In [ ]:
library_model = baseline_pipeline(l2=0.0).fit(X_train, y_train)
library_prediction = library_model.predict(X_valid)
display(
    pd.DataFrame(
        [
            {
                "модель": "Среднее train",
                **metrics(y_valid, np.full(len(y_valid), y_train.mean())),
            },
            {
                "модель": "Библиотечная линейная модель",
                **metrics(y_valid, library_prediction),
            },
        ]
    )
)

In [ ]:
assert VARIANT in (3, 4)

## Одна история, два разных места самостоятельной работы
**Вариант 3** реализует математический компонент: критерий/градиент и решение МНК → Ridge.
Оценка по фолдам у него выполняется готовыми библиотечными средствами.
**Вариант 4** использует готовую библиотечную модель, но самостоятельно управляет
обучением и оценкой на разбиениях. Он не реализует Ridge.

Вы делаете только назначенный маршрут. Совместное выполнение 3+4 **не требуется** для 10 баллов.

## Вариант 3. Часть A. МНК без обращения матрицы
`маркер=V3-A`

Реализуйте две функции. `ridge_value_gradient` возвращает число $J$ и градиент формы $(d+1,)$.
`solve_ridge` возвращает вектор $\theta$ той же формы. В части A требуется только `l2=0`;
параметр присутствует сразу, чтобы после лекции 3 не менять интерфейс.

Для решения разрешён `np.linalg.lstsq(Z, y, rcond=None)[0]` — собственные QR/SVD не нужны.
Явное вычисление обратной матрицы не требуется. При зависимых столбцах возможны разные
коэффициенты с одинаковыми прогнозами; корректность проверяем по прогнозу и критерию.

**Пример:** $Z=\begin{pmatrix}1&0\\1&1\\1&2\end{pmatrix}$, $y=(1,3,5)^\top$.
Решение $\theta=(1,2)^\top$: свободный член 1, наклон 2, MSE равна нулю.
Это пример для проверки, а не числа, которые нужно зашить в функцию.

In [ ]:
def ridge_value_gradient(Z, y, theta, l2=0.0):
    Z, y = check_arrays(Z, y)
    if l2 < 0:
        raise ValueError("l2>=0")
    # A: MSE и градиент. B: добавить штраф, но НЕ для theta[0].
    raise NotImplementedError("V3-A/B: критерий и градиент")


def solve_ridge(Z, y, l2=0.0):
    Z, y = check_arrays(Z, y)
    if l2 < 0:
        raise ValueError("l2>=0")
    # A: нерегуляризованное lstsq. B: расширенная задача со штрафом.
    raise NotImplementedError("V3-A/B: линейный решатель")

In [ ]:
def check_v3_basic():
    Z = augment(np.array([[0.0], [1.0], [2.0]]))
    y = np.array([1.0, 3.0, 5.0])
    theta = solve_ridge(Z, y, 0.0)
    np.testing.assert_allclose(theta, [1.0, 2.0], atol=1e-10)
    value, gradient = ridge_value_gradient(Z, y, np.zeros(2), 0.0)
    np.testing.assert_allclose(value, 35.0 / 3.0)
    np.testing.assert_allclose(gradient, [-6.0, -26.0 / 3.0])
    # Зависимые столбцы: сравниваем прогноз, а не конкретный выбор коэффициентов.
    Z_duplicate = np.column_stack([Z, Z[:, 1]])
    duplicate = solve_ridge(Z_duplicate, y, 0.0)
    np.testing.assert_allclose(Z_duplicate @ duplicate, y, atol=1e-9)


if VARIANT == 3:
    checks["A"] = run_checks("V3-A / МНК", check_v3_basic)

In [ ]:
class OwnRidge(RegressorMixin, BaseEstimator):
    # Внешняя обёртка готова. Внутри вызывается ваш solve_ridge.
    def __init__(self, l2=0.0):
        self.l2 = l2

    def fit(self, X, y):
        Z = augment(X)
        self.theta_ = np.asarray(solve_ridge(Z, np.asarray(y), self.l2))
        if self.theta_.shape != (Z.shape[1],):
            raise ValueError("Неверная форма theta")
        self.intercept_ = float(self.theta_[0])
        self.coef_ = self.theta_[1:].copy()
        self.n_features_in_ = Z.shape[1] - 1
        return self

    def predict(self, X):
        check_is_fitted(self, "theta_")
        return augment(X) @ self.theta_


def factory_v3(l2):
    return make_pipeline(make_preprocessor(), OwnRidge(l2=l2))

## Часть A: собственная часть уже участвует в решении
В маршруте 3 это ваш решатель; в маршруте 4 — ваша процедура оценки.
Идентичность отдельных чисел библиотечным нормальна: цель — реализовать правильную задачу,
а не любой ценой получить другую ошибку.

In [ ]:
basic_model = final_model = None
if checks.get("A", False):
    factory = factory_v3 if VARIANT == 3 else factory_v4
    basic_model = factory(0.0).fit(X_train, y_train)
    final_model = basic_model
    if VARIANT == 3:
        display(
            pd.DataFrame(
                [{"l2": 0.0, **metrics(y_valid, basic_model.predict(X_valid))}]
            )
        )
    else:
        display(holdout_scores(factory, [0.0], X_train, y_train, X_valid, y_valid))
else:
    print("Сквозной эксперимент ждёт часть A назначенного варианта.")

## Часть B: выбираем регуляризацию, не используя test
`маркер=B`

Наша нормировка: $J=\mathrm{MSE}+\lambda\|w\|^2$. У sklearn Ridge:
$\mathrm{SSE}+\alpha\|w\|^2$, поэтому $\alpha=n_{fit}\lambda$.
Внутри каждого фолда число $n_{fit}$ своё. Готовая библиотечная обёртка учитывает это.
Свободный член не штрафуем.

Подбираем $\lambda\in\{0,0.03,1\}$ на **трёх фолдах только исходного train**.
После выбора вновь обучаем цепочку на полном train и смотрим validation.
Ранее validation уже использовалась в части A; не называем её независимой финальной проверкой.
Закрытый test остаётся отдельным. Выбор по CV не делает выбранную CV-оценку несмещённой
независимой оценкой качества после перебора настроек.

### Вариант 3: расширяем математическую задачу
Возвращайтесь к двум функциям части A и добавьте регуляризацию. Градиент штрафа равен
$2\lambda(0,\theta_1,\ldots,\theta_d)^\top$.

Вместо обращения $Z^\top Z$ решите обычную задачу наименьших квадратов для
$$Z_{ext}=\begin{pmatrix}Z\\\sqrt{n\lambda}\,P\end{pmatrix},\quad
y_{ext}=\begin{pmatrix}y\\0\end{pmatrix},\quad P=\operatorname{diag}(0,1,\ldots,1).$$

**Пример:** у двух коэффициентов `(b,w)` матрица штрафа $P=\begin{pmatrix}0&0\\0&1\end{pmatrix}$.
Нулевая первая диагональ оставляет свободный член без штрафа.
Самостоятельно проверьте раскрытием нормы, что получилась исходная цель, умноженная на $n$.
Переход математически задан; задача — корректно превратить его в код общего размера.

In [ ]:
def check_v3_full():
    rng = np.random.default_rng(44)
    X = rng.normal(size=(17, 4))
    X[:, 3] = X[:, 1]
    Z = augment(X)
    y = rng.normal(size=17)
    theta = rng.normal(size=5)
    Z_before = Z.copy()
    y_before = y.copy()
    theta_before = theta.copy()
    eps = 1e-6
    for l2 in (0.03, 0.4):
        value, grad = ridge_value_gradient(Z, y, theta, l2)
        expected = np.mean((Z @ theta - y) ** 2) + l2 * np.sum(theta[1:] ** 2)
        np.testing.assert_allclose(value, expected)
        finite = np.zeros_like(theta)
        for j in range(len(theta)):
            direction = np.zeros_like(theta)
            direction[j] = eps
            finite[j] = (
                ridge_value_gradient(Z, y, theta + direction, l2)[0]
                - ridge_value_gradient(Z, y, theta - direction, l2)[0]
            ) / (2 * eps)
        np.testing.assert_allclose(grad, finite, rtol=1e-5, atol=1e-6)
        fitted = solve_ridge(Z, y, l2)
        ref = Ridge(alpha=len(y) * l2, solver="svd").fit(X, y)
        np.testing.assert_allclose(Z @ fitted, ref.predict(X), atol=1e-9)
        np.testing.assert_allclose(
            ridge_value_gradient(Z, y, fitted, l2)[1], 0.0, atol=1e-8
        )
        shifted = solve_ridge(Z, y + 100.0, l2)
        np.testing.assert_allclose(shifted[0], fitted[0] + 100.0, atol=1e-8)
        np.testing.assert_allclose(shifted[1:], fitted[1:], atol=1e-8)
    np.testing.assert_array_equal(Z, Z_before)
    np.testing.assert_array_equal(y, y_before)
    np.testing.assert_array_equal(theta, theta_before)


if VARIANT == 3:
    checks["B"] = run_checks("V3-B / Ridge", check_v3_full) if MODE == "full" else False

In [ ]:
def library_cv_for_v3(factory, strengths, X, y, folds):
    # Готовая библиотечная процедура, не ручной ответ варианта 4.
    rows, oof = [], {}
    for l2 in strengths:
        scores = cross_validate(
            factory(l2), X, y, cv=folds, scoring="neg_mean_squared_error"
        )
        predictions = cross_val_predict(factory(l2), X, y, cv=folds, method="predict")
        errors = -scores["test_score"]
        rows.append(
            {
                "l2": l2,
                "mean_fold_mse": float(errors.mean()),
                "std_fold_mse": float(errors.std(ddof=0)),
                "oof_mse": mse(y, predictions),
            }
        )
        oof[l2] = predictions
    return pd.DataFrame(rows), oof

## Общий завершающий эксперимент
До запуска запишите, какое значение регуляризации ожидаете более устойчивым и почему.
Изменяем только `l2`; данные, признаки, предобработка, фолды и метрика зафиксированы.
Выбор минимального `mean_fold_mse` выполняется готовой ячейкой; при точном равенстве
остаётся первое значение в заранее объявленном списке.

OOF-прогнозы каждой модели сделаны без обучения на соответствующей строке, но выбор
лучшей настройки по этим результатам всё равно использует OOF-ответы. Это не «новый test».

In [ ]:
hypothesis = ""  # Ожидание до запуска.

In [ ]:
experiment = oof_predictions = None
if MODE == "full" and checks.get("A", False) and checks.get("B", False):
    strengths = [0.0, 0.03, 1.0]
    folds = list(KFold(n_splits=3, shuffle=True, random_state=23).split(X_train))
    evaluator = library_cv_for_v3 if VARIANT == 3 else cv_scores
    experiment, oof_predictions = evaluator(factory, strengths, X_train, y_train, folds)
    display(experiment)
    best_l2 = float(experiment.loc[experiment["mean_fold_mse"].idxmin(), "l2"])
    final_model = factory(best_l2).fit(X_train, y_train)
    display(
        pd.DataFrame(
            [{"выбрано l2": best_l2, **metrics(y_valid, final_model.predict(X_valid))}]
        )
    )
    fig, ax = plt.subplots(figsize=(7.2, 4.0))
    ax.errorbar(
        np.arange(len(strengths)),
        experiment["mean_fold_mse"],
        yerr=experiment["std_fold_mse"],
        marker="o",
        capsize=4,
        color=BLUE,
    )
    ax.set(
        xticks=np.arange(len(strengths)),
        xticklabels=[str(v) for v in strengths],
        xlabel="lambda",
        ylabel="MSE, мин²",
        title="CV: среднее ± описательный разброс фолдов",
    )
    ax.grid(alpha=0.2)
    fig.tight_layout()
    plt.show()
    plot_residuals(
        train,
        y_train,
        oof_predictions[best_l2],
        title="OOF-остатки выбранной конфигурации",
    )
else:
    print("Часть B не выполнена. Базовый маршрут возможен после готовой части A.")

**Содержательный вывод.** Различите три события: правильное решение оптимизационной задачи,
выбор настройки по CV и оценка выбранной процедуры на новых данных. Корректность вашего кода
доказывается не только минимальным числом в таблице, а отдельными тестами контракта.
Если линейная модель оставляет закономерность в остатках, это может быть ограничением
представления, а не ошибкой решателя или поводом бесконечно уменьшать learning rate.

## finally

1. Что реализовано мной и какие ограничения остались?
2. Что сравнивалось, что было зафиксировано, какую метрику использовали?
3. Какой результат получился **с числами** и совпал ли с ожиданием?
4. Где именно проходил `fit` и какие данные участвовали в выборе настроек?
5. Почему результат не доказывает качество на всех будущих обращениях?

Пройдите `Restart Kernel → Run All`. Статус незавершённой части допустим только для
явно выбранного базового маршрута; библиотечный baseline не заменяет вашу модель.

In [ ]:
conclusion = ""  # 4–8 предложений по вопросам выше.
print("Вывод заполнен:", bool(conclusion.strip()))
print("Проверки:", checks)
print("Заявленный маршрут:", MODE, "| максимум маршрута:", 5 if MODE == "base" else 10)

### Одна финальная проверка
Сначала закончите выбор настроек. Только затем поставьте `EXPORT_FINAL=True` и выполните
ячейку ниже.
**Низкая test-ошибка не является порогом получения баллов.**

In [ ]:
if EXPORT_FINAL:
    if final_model is None:
        raise RuntimeError("Сначала получите собственную работающую final_model.")
    export_predictions(
        ROOT,
        test,
        final_model.predict(test[FEATURES]),
        f"lab1_v{VARIANT}_predictions.csv",
    )
else:
    print("Test не использован: EXPORT_FINAL=False.")

## Справка
Ссылки нужны для дальнейшего чтения; выполнение notebook не требует интернета.
- [NumPy: `lstsq`](https://numpy.org/doc/stable/reference/generated/numpy.linalg.lstsq.html)
- [scikit-learn: SGD](https://scikit-learn.org/stable/modules/sgd.html)
- [scikit-learn: ошибки предобработки и утечки](https://scikit-learn.org/stable/common_pitfalls.html)

Внешний API похож на scikit-learn, но учебный solver не является полной копией `SGDRegressor`:
отличаются инициализация, нормировка критерия, расписание шага и критерии остановки.